# UC03 — Fraude en Solicitudes e Identidad

Detección de identidades sintéticas en el momento de contratación de póliza.

**Valor de negocio**: Bloquear solicitudes fraudulentas antes de emitir la póliza.

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS FRAUDE_IDENTIDAD_DB;
USE SCHEMA FRAUDE_IDENTIDAD_DB.PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Solicitudes de Póliza

1.000 solicitudes con datos de identidad. 15% son identidades sintéticas.

In [ ]:
CREATE OR REPLACE TABLE solicitudes AS
SELECT
    'SOL' || LPAD(SEQ4(), 5, '0') AS solicitud_id,
    UNIFORM(20, 70, RANDOM()) AS edad,
    UNIFORM(6, 360, RANDOM()) AS antiguedad_empleo_meses,
    UNIFORM(15000, 80000, RANDOM())::DECIMAL(10,2) AS ingresos_declarados,
    UNIFORM(0, 8, RANDOM()) AS num_consultas_bureau_6m,
    CASE WHEN UNIFORM(0,1,RANDOM()) < 0.15 THEN 1 ELSE 0 END AS es_identidad_sintetica,
    CASE WHEN es_identidad_sintetica=1 AND UNIFORM(0,1,RANDOM())<0.7 THEN 0 ELSE 1 END AS tiene_historial_crediticio,
    CASE WHEN es_identidad_sintetica=1 AND UNIFORM(0,1,RANDOM())<0.8 THEN 1 ELSE 0 END AS email_temporal,
    CASE WHEN es_identidad_sintetica=1 AND UNIFORM(0,1,RANDOM())<0.6 THEN 1 ELSE 0 END AS telefono_prepago,
    CASE WHEN es_identidad_sintetica=1 AND UNIFORM(0,1,RANDOM())<0.5 THEN 0 ELSE 1 END AS direccion_coincide_bureau
FROM TABLE(GENERATOR(ROWCOUNT => 1000));

SELECT * FROM solicitudes LIMIT 10;

---

## Paso 3: Generar Señales de Comportamiento Online

In [ ]:
CREATE OR REPLACE TABLE sesion_solicitud AS
SELECT
    s.solicitud_id,
    CASE WHEN s.es_identidad_sintetica=1 THEN UNIFORM(15,55,RANDOM()) ELSE UNIFORM(120,600,RANDOM()) END AS tiempo_formulario_seg,
    CASE WHEN s.es_identidad_sintetica=1 THEN UNIFORM(0,2,RANDOM()) ELSE UNIFORM(2,8,RANDOM()) END AS num_correcciones,
    CASE WHEN s.es_identidad_sintetica=1 THEN UNIFORM(2,5,RANDOM()) ELSE UNIFORM(9,21,RANDOM()) END AS hora_solicitud,
    CASE WHEN s.es_identidad_sintetica=1 AND UNIFORM(0,1,RANDOM())<0.3 THEN 1 ELSE CASE WHEN UNIFORM(0,1,RANDOM())<0.85 THEN 1 ELSE 0 END END AS dispositivo_conocido,
    CASE WHEN s.es_identidad_sintetica=1 AND UNIFORM(0,1,RANDOM())<0.7 THEN 1 ELSE 0 END AS ip_vpn,
    CASE WHEN s.es_identidad_sintetica=1 THEN UNIFORM(3,10,RANDOM()) ELSE UNIFORM(0,2,RANDOM()) END AS solicitudes_misma_ip_24h
FROM solicitudes s;

SELECT * FROM sesion_solicitud LIMIT 10;

---

## Paso 4: Feature Engineering

In [ ]:
CREATE OR REPLACE TABLE features_identidad AS
SELECT
    s.solicitud_id,
    s.edad::FLOAT AS edad,
    s.antiguedad_empleo_meses::FLOAT AS antiguedad_empleo,
    s.num_consultas_bureau_6m::FLOAT AS consultas_bureau,
    s.tiene_historial_crediticio::FLOAT AS tiene_historial,
    s.email_temporal::FLOAT AS email_temporal,
    s.telefono_prepago::FLOAT AS telefono_prepago,
    s.direccion_coincide_bureau::FLOAT AS direccion_coincide,
    ss.tiempo_formulario_seg::FLOAT AS tiempo_formulario,
    ss.num_correcciones::FLOAT AS num_correcciones,
    ss.hora_solicitud::FLOAT AS hora_solicitud,
    ss.dispositivo_conocido::FLOAT AS dispositivo_conocido,
    ss.ip_vpn::FLOAT AS ip_vpn,
    ss.solicitudes_misma_ip_24h::FLOAT AS solicitudes_misma_ip,
    s.es_identidad_sintetica
FROM solicitudes s JOIN sesion_solicitud ss ON s.solicitud_id = ss.solicitud_id;

SELECT * FROM features_identidad LIMIT 10;

---

## Paso 5: Train/Test Split

In [ ]:
CREATE OR REPLACE TABLE entrenamiento AS SELECT * FROM features_identidad SAMPLE (80);
CREATE OR REPLACE TABLE test AS SELECT * FROM features_identidad WHERE solicitud_id NOT IN (SELECT solicitud_id FROM entrenamiento);

SELECT 'Entrenamiento' AS conjunto, COUNT(*) AS total, SUM(es_identidad_sintetica) AS sinteticas FROM entrenamiento
UNION ALL SELECT 'Test', COUNT(*), SUM(es_identidad_sintetica) FROM test;

---

## Paso 6: Entrenar Detector de Identidad Sintética

In [ ]:
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION detector_identidad(
    INPUT_DATA     => SYSTEM$REFERENCE('TABLE', 'entrenamiento'),
    TARGET_COLNAME => 'es_identidad_sintetica',
    CONFIG_OBJECT  => {'evaluate': TRUE}
);

---

## Paso 7: Evaluar

In [ ]:
CALL detector_identidad!SHOW_EVALUATION_METRICS();

In [ ]:
CALL detector_identidad!SHOW_CONFUSION_MATRIX();

In [ ]:
CALL detector_identidad!SHOW_FEATURE_IMPORTANCE();

---

## Paso 8: Puntuar Solicitudes

- **Bloquear** (≥80%): Rechazar automáticamente
- **Verificación Manual** (40-80%): Solicitar documentación
- **Aprobar** (<40%): Continuar contratación

In [ ]:
CREATE OR REPLACE TABLE predicciones_identidad AS
SELECT solicitud_id,
    detector_identidad!PREDICT(OBJECT_CONSTRUCT(
        'edad',edad,'antiguedad_empleo',antiguedad_empleo,'consultas_bureau',consultas_bureau,
        'tiene_historial',tiene_historial,'email_temporal',email_temporal,'telefono_prepago',telefono_prepago,
        'direccion_coincide',direccion_coincide,'tiempo_formulario',tiempo_formulario,
        'num_correcciones',num_correcciones,'hora_solicitud',hora_solicitud,
        'dispositivo_conocido',dispositivo_conocido,'ip_vpn',ip_vpn,'solicitudes_misma_ip',solicitudes_misma_ip
    )) AS pred,
    pred['class']::INT AS sintetica_pred,
    pred['probability']['1']::FLOAT AS prob_sintetica,
    CASE WHEN pred['probability']['1']::FLOAT>=0.80 THEN 'Bloquear'
         WHEN pred['probability']['1']::FLOAT>=0.40 THEN 'Verificación Manual'
         ELSE 'Aprobar' END AS decision,
    es_identidad_sintetica AS sintetica_real
FROM test;

SELECT solicitud_id, ROUND(prob_sintetica,3) AS prob, decision, sintetica_real
FROM predicciones_identidad ORDER BY prob_sintetica DESC LIMIT 20;

---

## Paso 9: Dashboard

In [ ]:
import pandas as pd

session = get_active_session()
st.title('Fraude en Solicitudes e Identidad')
st.markdown('### Detección de Identidades Sintéticas — Snowflake Cortex')

df = session.table('predicciones_identidad').to_pandas()
c1,c2,c3,c4 = st.columns(4)
with c1: st.metric('Total Solicitudes', len(df))
with c2: st.metric('Bloqueadas', len(df[df['DECISION']=='Bloquear']))
with c3: st.metric('Exactitud', f"{(df['SINTETICA_PRED']==df['SINTETICA_REAL']).mean():.1%}")
with c4: st.metric('Verificación Manual', len(df[df['DECISION']=='Verificación Manual']))

st.markdown('---')
st.subheader('Distribución de Decisiones')
rc = df['DECISION'].value_counts()
st.bar_chart(pd.DataFrame({'Solicitudes': rc.values}, index=rc.index))

st.markdown('---')
st.subheader('Solicitudes por Decisión')
filtro = st.multiselect('Filtrar', ['Bloquear','Verificación Manual','Aprobar'], default=['Bloquear','Verificación Manual'])
fdf = df[df['DECISION'].isin(filtro)].sort_values('PROB_SINTETICA', ascending=False)
fdf['Prob %'] = fdf['PROB_SINTETICA'].apply(lambda x: f'{x:.1%}')
st.dataframe(fdf[['SOLICITUD_ID','Prob %','DECISION','SINTETICA_PRED','SINTETICA_REAL']], use_container_width=True, height=400)
st.caption('Desarrollado con Snowflake Cortex ML + Streamlit')

---

## Paso 10: Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS FRAUDE_IDENTIDAD_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;